# Chapter 10 Companion Notebook: Portfolio Theory and Asset Allocation

This notebook reproduces every worked numerical example from Chapter 10 of *AI in Finance* (`chapter10.tex`): the two-asset minimum-variance portfolio, three-asset matrix-form optimization and covariance shrinkage, CAPM beta estimation and risk decomposition, Sharpe ratios and the tangency (maximum-Sharpe) portfolio, a two-factor model, a head-to-head test of ridge regression versus OLS on collinear factor exposures, risk parity, the long-only constraint example, tracking error/information ratio, the currency and rebalancing-cost examples, and the robo-advisor examples (risk-tolerance heuristic vs. mean-variance, tax-loss harvesting, fee drag).

---

**© 2026 Wulin Suo. All rights reserved.** This notebook is a companion to the draft manuscript *AI in Finance* and is provided for personal, educational use. No part of this notebook may be reproduced, distributed, or transmitted in any form or by any means without the prior written permission of the author, except for brief quotations in a review. Contact: Wulin.Suo@Queensu.ca

## 1. Diversification and the minimum-variance portfolio (Section 10.2)

In [1]:
import numpy as np
import pandas as pd

var_A, var_B, cov_AB = 0.000414, 0.000118, -0.000198
std_A, std_B = np.sqrt(var_A), np.sqrt(var_B)
corr_AB = cov_AB / (std_A * std_B)
print(f"std_A={std_A:.4%}, std_B={std_B:.4%}, corr_AB={corr_AB:.4f}")

wA_mv = (var_B - cov_AB) / (var_A + var_B - 2 * cov_AB)
wB_mv = 1 - wA_mv
print(f"Minimum-variance weights: wA={wA_mv:.4%}, wB={wB_mv:.4%}")

port_var_mv = wA_mv**2 * var_A + wB_mv**2 * var_B + 2 * wA_mv * wB_mv * cov_AB
print(f"Minimum-variance portfolio std: {np.sqrt(port_var_mv):.4%}")

# compare to equal-weight (from Chapter 2)
port_var_eq = 0.25 * var_A + 0.25 * var_B + 2 * 0.25 * cov_AB
print(f"Equal-weight portfolio std: {np.sqrt(port_var_eq):.4%}")

std_A=2.0347%, std_B=1.0863%, corr_AB=-0.8958
Minimum-variance weights: wA=34.0517%, wB=65.9483%
Minimum-variance portfolio std: 0.3224%
Equal-weight portfolio std: 0.5831%


## 2. Mean-variance optimization in matrix form (Section 10.4)

In [2]:
Sigma3 = np.array([
    [0.000414, -0.000198, -0.000383],
    [-0.000198, 0.000118, 0.000176],
    [-0.000383, 0.000176, 0.000358],
])

ones = np.ones(3)
Sigma3_inv = np.linalg.inv(Sigma3)
w_mv3 = Sigma3_inv @ ones / (ones @ Sigma3_inv @ ones)
print(f"Three-asset minimum-variance weights: {w_mv3}")

port_var_mv3 = w_mv3 @ Sigma3 @ w_mv3
print(f"Three-asset min-variance std: {np.sqrt(port_var_mv3):.4%}")

w_eq3 = np.array([1/3, 1/3, 1/3])
port_var_eq3 = w_eq3 @ Sigma3 @ w_eq3
print(f"Three-asset equal-weight std (check vs Chapter 2's 0.0030): {np.sqrt(port_var_eq3):.4f}")

Three-asset minimum-variance weights: [0.44785178 0.14224515 0.40990308]
Three-asset min-variance std: 0.0503%
Three-asset equal-weight std (check vs Chapter 2's 0.0030): 0.0030


### Covariance shrinkage (Section 10.4.1)

In [3]:
avg_var = np.trace(Sigma3) / 3
F = np.eye(3) * avg_var
delta = 0.5
Sigma_shrunk = delta * F + (1 - delta) * Sigma3
print(f"Average variance: {avg_var:.6f}")
print(f"Shrunk covariance matrix:\n{Sigma_shrunk}")

Sigma_shrunk_inv = np.linalg.inv(Sigma_shrunk)
w_shrunk = Sigma_shrunk_inv @ ones / (ones @ Sigma_shrunk_inv @ ones)
print(f"Shrunk minimum-variance weights: {w_shrunk}")

port_var_shrunk_true = w_shrunk @ Sigma3 @ w_shrunk
print(f"Shrunk portfolio's true (sample) std: {np.sqrt(port_var_shrunk_true):.4%}")

Average variance: 0.000297
Shrunk covariance matrix:
[[ 3.55333333e-04 -9.90000000e-05 -1.91500000e-04]
 [-9.90000000e-05  2.07333333e-04  8.80000000e-05]
 [-1.91500000e-04  8.80000000e-05  3.27333333e-04]]
Shrunk minimum-variance weights: [0.39201467 0.30459982 0.30338552]
Shrunk portfolio's true (sample) std: 0.1290%


## 3. CAPM and beta estimation (Section 10.5)

In [4]:
market = np.array([-2.0, -1.0, 0.0, 1.0, 2.0, 3.0]) / 100
asset = np.array([-2.5, -1.3, 0.2, 1.1, 2.6, 3.3]) / 100

mbar, abar = market.mean(), asset.mean()
beta = np.sum((market - mbar) * (asset - abar)) / np.sum((market - mbar) ** 2)
alpha = abar - beta * mbar
fitted = alpha + beta * market
resid = asset - fitted
r2 = 1 - np.sum(resid**2) / np.sum((asset - abar) ** 2)

print(f"beta = {beta:.4f}, alpha = {alpha:.4%}, R^2 = {r2:.4f}")

rf, mrp = 0.04, 0.06
re_capm = rf + beta * mrp
print(f"CAPM cost of equity: {re_capm:.4%} (compare to Chapter 5's assumed 11.2% using beta=1.2)")

beta = 1.1886, alpha = -0.0276%, R^2 = 0.9923
CAPM cost of equity: 11.1314% (compare to Chapter 5's assumed 11.2% using beta=1.2)


### Systematic vs. idiosyncratic risk (Section 10.5.1)

In [5]:
var_m = market.var(ddof=1)
var_total = asset.var(ddof=1)
systematic_var = beta**2 * var_m
idio_var = var_total - systematic_var

print(f"Var(market) = {var_m:.6f}")
print(f"Systematic variance = {systematic_var:.6f}")
print(f"Total variance = {var_total:.6f}")
print(f"Idiosyncratic variance = {idio_var:.8f}")
print(f"Systematic fraction (should equal R^2): {systematic_var/var_total:.4f}")

Var(market) = 0.000350
Systematic variance = 0.000494
Total variance = 0.000498
Idiosyncratic variance = 0.00000382
Systematic fraction (should equal R^2): 0.9923


## 4. Sharpe ratios (Section 10.6)

In [6]:
mean_A, mean_B = 0.0076, 0.0100
mean_eq = 0.5 * mean_A + 0.5 * mean_B
std_eq = np.sqrt(port_var_eq)
mean_mv = wA_mv * mean_A + wB_mv * mean_B
std_mv = np.sqrt(port_var_mv)

sharpe_eq = mean_eq / std_eq
sharpe_mv = mean_mv / std_mv
print(f"Equal-weight: mean={mean_eq:.4%}, std={std_eq:.4%}, Sharpe={sharpe_eq:.2f}")
print(f"Min-variance:  mean={mean_mv:.4%}, std={std_mv:.4%}, Sharpe={sharpe_mv:.2f}")

Equal-weight: mean=0.8800%, std=0.5831%, Sharpe=1.51
Min-variance:  mean=0.9183%, std=0.3224%, Sharpe=2.85


## 4b. The tangency (maximum-Sharpe) portfolio (Section 10.6.1)

Compute the tangency portfolio for AssetA/AssetB (rf~0, where it nearly coincides with minimum-variance) and contrast with the robo-advisor equity/bond example (rf=2%, where it does not).

In [7]:
Sigma_AB = np.array([[var_A, cov_AB], [cov_AB, var_B]])
mu_AB = np.array([mean_A, mean_B])
rf_AB = 0.0

Sigma_AB_inv = np.linalg.inv(Sigma_AB)
raw_tan = Sigma_AB_inv @ (mu_AB - rf_AB)
w_tan_AB = raw_tan / raw_tan.sum()
ret_tan_AB = w_tan_AB @ mu_AB
std_tan_AB = np.sqrt(w_tan_AB @ Sigma_AB @ w_tan_AB)
sharpe_tan_AB = (ret_tan_AB - rf_AB) / std_tan_AB

print(f"AssetA/AssetB tangency weights: wA={w_tan_AB[0]:.4%}, wB={w_tan_AB[1]:.4%}")
print(f"Tangency Sharpe: {sharpe_tan_AB:.3f} (vs. min-variance Sharpe: {sharpe_mv:.3f})")

# Contrast: Section 10.14's equity/bond assumptions, with a meaningful risk-free rate
sigma_e_t, sigma_b_t, rho_eb_t = 0.16, 0.05, -0.20
cov_eb_t = rho_eb_t * sigma_e_t * sigma_b_t
r_e_t, r_b_t = 0.08, 0.03
rf_eb = 0.02

Sigma_eb = np.array([[sigma_e_t**2, cov_eb_t], [cov_eb_t, sigma_b_t**2]])
mu_eb = np.array([r_e_t, r_b_t])

w_e_mv_t = (sigma_b_t**2 - cov_eb_t) / (sigma_e_t**2 + sigma_b_t**2 - 2 * cov_eb_t)
w_b_mv_t = 1 - w_e_mv_t
sigma_mv_t = np.sqrt(w_e_mv_t**2 * sigma_e_t**2 + w_b_mv_t**2 * sigma_b_t**2 + 2 * w_e_mv_t * w_b_mv_t * cov_eb_t)
ret_mv_t = w_e_mv_t * r_e_t + w_b_mv_t * r_b_t
sharpe_mv_eb = (ret_mv_t - rf_eb) / sigma_mv_t

Sigma_eb_inv = np.linalg.inv(Sigma_eb)
raw_tan_eb = Sigma_eb_inv @ (mu_eb - rf_eb)
w_tan_eb = raw_tan_eb / raw_tan_eb.sum()
ret_tan_eb = w_tan_eb @ mu_eb
std_tan_eb = np.sqrt(w_tan_eb @ Sigma_eb @ w_tan_eb)
sharpe_tan_eb = (ret_tan_eb - rf_eb) / std_tan_eb

print(f"\nEquity/bond min-variance weights: w_e={w_e_mv_t:.4%}, w_b={w_b_mv_t:.4%}")
print(f"Equity/bond tangency weights:     w_e={w_tan_eb[0]:.4%}, w_b={w_tan_eb[1]:.4%}")
print(f"Equity/bond tangency Sharpe: {sharpe_tan_eb:.3f} (vs. min-variance Sharpe at rf=2%: {sharpe_mv_eb:.3f})")

AssetA/AssetB tangency weights: wA=33.7589%, wB=66.2411%
Tangency Sharpe: 2.849 (vs. min-variance Sharpe: 2.848)

Equity/bond min-variance weights: w_e=13.0990%, w_b=86.9010%
Equity/bond tangency weights:     w_e=32.0463%, w_b=67.9537%
Equity/bond tangency Sharpe: 0.468 (vs. min-variance Sharpe at rf=2%: 0.374)


## 5. Multi-factor model (Section 10.7)

In [8]:
smb = np.array([1.0, 0.5, -0.3, 0.2, -0.5, 0.8]) / 100
true_alpha, beta_mkt, beta_smb = -0.0003, 1.2, 0.4
noise = np.array([0.0005, -0.0008, 0.0010, -0.0005, 0.0003, -0.0002])
factor_asset = np.round(true_alpha + beta_mkt * market + beta_smb * smb + noise, 4)
print(f"Constructed asset returns: {factor_asset}")

X = np.column_stack([np.ones(6), market, smb])
beta_hat = np.linalg.inv(X.T @ X) @ X.T @ factor_asset
fitted_factor = X @ beta_hat
r2_factor = 1 - np.sum((factor_asset - fitted_factor) ** 2) / np.sum((factor_asset - factor_asset.mean()) ** 2)

print(f"alpha={beta_hat[0]:.4%}, beta_MKT={beta_hat[1]:.4f}, beta_SMB={beta_hat[2]:.4f}")
print(f"R^2 = {r2_factor:.4f}")

Constructed asset returns: [-0.0198 -0.0111 -0.0005  0.012   0.022   0.0387]
alpha=-0.0065%, beta_MKT=1.1903, beta_SMB=0.3517
R^2 = 0.9992


## 5b. Machine learning for factor exposures: ridge regression vs. OLS (Section 10.12.1)

The two-factor model above only has 6 months of data and 2 factors, too little collinearity to show a real problem. Here we extend to 12 months and 5 candidate factors, one of which ("quality") is deliberately constructed to be highly collinear with two others, exactly the kind of near-multicollinearity real factor libraries run into, and compare OLS to ridge regression at recovering the *known* true factor exposures.

In [9]:
from sklearn.linear_model import Ridge, LinearRegression

rng_factor = np.random.default_rng(3)
n_months = 12

mkt_f = rng_factor.normal(0.008, 0.04, n_months)
smb_f = rng_factor.normal(0.001, 0.02, n_months)
hml_f = rng_factor.normal(0.000, 0.02, n_months)
mom_f = rng_factor.normal(0.002, 0.025, n_months)
# "quality" is constructed as a near-linear combination of HML and momentum
quality_f = 0.6 * hml_f + 0.5 * mom_f + rng_factor.normal(0, 0.005, n_months)

X_factors = np.column_stack([mkt_f, smb_f, hml_f, mom_f, quality_f])
true_beta_factors = np.array([1.1, 0.3, 0.2, -0.1, 0.15])
noise_factors = rng_factor.normal(0, 0.01, n_months)
y_factors = 0.001 + X_factors @ true_beta_factors + noise_factors

print(f"Condition number of X'X: {np.linalg.cond(X_factors.T @ X_factors):.1f}")
print(f"Correlation(HML, Quality): {np.corrcoef(hml_f, quality_f)[0,1]:.4f}")
print(f"Correlation(Momentum, Quality): {np.corrcoef(mom_f, quality_f)[0,1]:.4f}")

ols_factor = LinearRegression().fit(X_factors, y_factors)
ridge_factor = Ridge(alpha=0.005).fit(X_factors, y_factors)

print(f"\nTrue exposures (MKT, SMB, HML, MOM, Quality): {true_beta_factors}")
print(f"OLS estimated exposures:   {np.round(ols_factor.coef_, 3)}")
print(f"Ridge estimated exposures: {np.round(ridge_factor.coef_, 3)}")

ols_rmse = np.sqrt(np.mean((ols_factor.coef_ - true_beta_factors)**2))
ridge_rmse = np.sqrt(np.mean((ridge_factor.coef_ - true_beta_factors)**2))
print(f"\nOLS coefficient RMSE vs. true exposures: {ols_rmse:.4f}")
print(f"Ridge coefficient RMSE vs. true exposures: {ridge_rmse:.4f}")
print(f"Ridge improvement: {(1 - ridge_rmse/ols_rmse)*100:.1f}%")

Condition number of X'X: 709.8
Correlation(HML, Quality): 0.6104
Correlation(Momentum, Quality): 0.8917

True exposures (MKT, SMB, HML, MOM, Quality): [ 1.1   0.3   0.2  -0.1   0.15]
OLS estimated exposures:   [ 1.078 -0.467  0.979  1.435 -2.322]
Ridge estimated exposures: [ 0.957  0.124  0.027 -0.011 -0.075]

OLS coefficient RMSE vs. true exposures: 1.3901
Ridge coefficient RMSE vs. true exposures: 0.1674
Ridge improvement: 88.0%


## 6. Risk parity (Section 10.8)

In [10]:
w_rp_A = (1 / std_A) / (1 / std_A + 1 / std_B)
w_rp_B = 1 - w_rp_A
mean_rp = w_rp_A * mean_A + w_rp_B * mean_B
var_rp = w_rp_A**2 * var_A + w_rp_B**2 * var_B + 2 * w_rp_A * w_rp_B * cov_AB
std_rp = np.sqrt(var_rp)
sharpe_rp = mean_rp / std_rp

print(f"Risk parity weights: wA={w_rp_A:.4%}, wB={w_rp_B:.4%}")
print(f"Risk parity: mean={mean_rp:.4%}, std={std_rp:.4%}, Sharpe={sharpe_rp:.2f}")

Risk parity weights: wA=34.8057%, wB=65.1943%
Risk parity: mean=0.9165%, std=0.3233%, Sharpe=2.84


## 7. Long-only constraint example (Section 10.10)

In [11]:
sigX, sigY, rho_XY = 0.08, 0.20, 0.9
sigXY = rho_XY * sigX * sigY
varX, varY = sigX**2, sigY**2

wX_mv = (varY - sigXY) / (varX + varY - 2 * sigXY)
wY_mv = 1 - wX_mv
print(f"Unconstrained weights: wX={wX_mv:.4%}, wY={wY_mv:.4%}")

port_var_unc = wX_mv**2 * varX + wY_mv**2 * varY + 2 * wX_mv * wY_mv * sigXY
print(f"Unconstrained portfolio std: {np.sqrt(port_var_unc):.4%}")
print(f"Long-only corner solution (wX=100%) std: {sigX:.4%}")

Unconstrained weights: wX=145.4545%, wY=-45.4545%
Unconstrained portfolio std: 5.2570%
Long-only corner solution (wX=100%) std: 8.0000%


## 7b. Solving the constrained problem numerically (Section 10.10)

First, a demonstration of the numerical-tolerance pitfall itself: calling `scipy.optimize.minimize` on the raw (tiny) portfolio-variance objective terminates after a single iteration, incorrectly reporting success at the starting point.

In [12]:
import numpy as np
import scipy.optimize as sco

Sigma3 = np.array([
    [0.000414, -0.000198, -0.000383],
    [-0.000198, 0.000118, 0.000176],
    [-0.000383, 0.000176, 0.000358],
])
n3 = 3
x0_3 = np.array([1 / n3] * n3)
budget_constraint = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1},)
bounds_longonly = tuple((0.0, 1.0) for _ in range(n3))

def portfolio_var_raw(w, Sigma):
    return w @ Sigma @ w

res_naive = sco.minimize(portfolio_var_raw, x0_3, args=(Sigma3,), method='SLSQP',
                          bounds=bounds_longonly, constraints=budget_constraint)
print("naive (unscaled) result:", res_naive.x.round(4), " iterations:", res_naive.nit,
      " success:", res_naive.success)
print("(this incorrectly stays at the equal-weight starting point)")

naive (unscaled) result: [0.3333 0.3333 0.3333]  iterations: 1  success: True
(this incorrectly stays at the equal-weight starting point)


Rescaling the objective (working in squared-percent units) fixes this, and recovers the known closed-form minimum-variance weights.

In [13]:
def portfolio_var(w, Sigma):
    return (w @ Sigma @ w) * 1e6  # squared-percent units

res_longonly = sco.minimize(portfolio_var, x0_3, args=(Sigma3,), method='SLSQP',
                             bounds=bounds_longonly, constraints=budget_constraint)
print("rescaled result:", res_longonly.x.round(4), " iterations:", res_longonly.nit)
print("volatility:", np.sqrt(res_longonly.fun / 1e6))

ones = np.ones(3)
Sinv = np.linalg.inv(Sigma3)
w_closed = Sinv @ ones / (ones @ Sinv @ ones)
print("closed-form check:", w_closed.round(4), " volatility:", np.sqrt(w_closed @ Sigma3 @ w_closed))

rescaled result: [0.4479 0.1422 0.4099]  iterations: 5
volatility: 0.0005032071427913235
closed-form check: [0.4479 0.1422 0.4099]  volatility: 0.0005032071427912332


Validating the numerical method against the AssetX/AssetY long-only corner solution derived by hand in the text.

In [14]:
sigma_X, sigma_Y, rho_XY = 0.08, 0.20, 0.9
Sigma_XY = np.array([
    [sigma_X**2, rho_XY * sigma_X * sigma_Y],
    [rho_XY * sigma_X * sigma_Y, sigma_Y**2],
])
bounds_xy = tuple((0.0, 1.0) for _ in range(2))
res_xy = sco.minimize(portfolio_var, [0.5, 0.5], args=(Sigma_XY,), method='SLSQP',
                       bounds=bounds_xy, constraints=budget_constraint)
print("weights:", res_xy.x.round(4), " iterations:", res_xy.nit)
print("volatility:", np.sqrt(res_xy.fun / 1e6))

weights: [1. 0.]  iterations: 2
volatility: 0.08


A 40% position cap on the three-asset minimum-variance portfolio, and one point on the resulting constrained efficient frontier at a target return of 0.75%.

In [15]:
cap = 0.40
bounds_capped = tuple((0.0, cap) for _ in range(n3))
res_capped = sco.minimize(portfolio_var, x0_3, args=(Sigma3,), method='SLSQP',
                           bounds=bounds_capped, constraints=budget_constraint)
print(f"capped-at-{cap:.0%} weights:", res_capped.x.round(4))
print("capped volatility:", np.sqrt(res_capped.fun / 1e6))
print("uncapped volatility (for comparison):", np.sqrt(res_longonly.fun / 1e6))
print("ratio:", np.sqrt(res_capped.fun / 1e6) / np.sqrt(res_longonly.fun / 1e6))

a = np.array([100.00, 102.00, 101.00, 104.00, 103.00])
b = np.array([50.00, 50.50, 51.26, 51.00, 52.02])
c = np.array([200.00, 198.00, 202.00, 199.00, 203.00])
ra, rb, rc = np.diff(a) / a[:-1], np.diff(b) / b[:-1], np.diff(c) / c[:-1]
mean_returns_3 = np.array([ra.mean(), rb.mean(), rc.mean()])

def port_return(w):
    return w @ mean_returns_3

target = 0.0075
constraints_target = (
    {'type': 'eq', 'fun': lambda w: np.sum(w) - 1},
    {'type': 'eq', 'fun': lambda w: port_return(w) - target},
)
res_target = sco.minimize(portfolio_var, x0_3, args=(Sigma3,), method='SLSQP',
                           bounds=bounds_capped, constraints=constraints_target)
print("\nweights at target return", target, ":", res_target.x.round(4))
print("resulting volatility:", np.sqrt(res_target.fun / 1e6))
print("achieved return:", port_return(res_target.x))
print("success:", res_target.success)

capped-at-40% weights: [0.4    0.2839 0.3161]
capped volatility: 0.0011347871762941862
uncapped volatility (for comparison): 0.0005032071427913235
ratio: 2.255109436641631

weights at target return 0.0075 : [0.4    0.3513 0.2487]
resulting volatility: 0.0013606659762317262
achieved return: 0.007499999999479322
success: True


## 8. Tracking error and information ratio (Section 10.11)

In [16]:
port_returns = np.array([0.0150, 0.0026, 0.0123, 0.0052])  # equal-weight AssetA/B, Chapter 2
bench_returns = np.array([0.0100, 0.0150, -0.0051, 0.0200])  # AssetB as benchmark proxy

active = port_returns - bench_returns
te = active.std(ddof=1)
mean_active = active.mean()
info_ratio = mean_active / te

print(f"Active returns: {active}")
print(f"Tracking error: {te:.4%}")
print(f"Mean active return: {mean_active:.4%}")
print(f"Information ratio: {info_ratio:.4f}")

Active returns: [ 0.005  -0.0124  0.0174 -0.0148]
Tracking error: 1.5218%
Mean active return: -0.1200%
Information ratio: -0.0789


## 9. Currency risk (Section 10.13)

In [17]:
local_return = 0.05
fx_change = -0.03
unhedged = (1 + local_return) * (1 + fx_change) - 1
print(f"Unhedged USD return: {unhedged:.4%}")
print(f"Hedged USD return (approx, ignoring hedge cost): {local_return:.4%}")

Unhedged USD return: 1.8500%
Hedged USD return (approx, ignoring hedge cost): 5.0000%


## 10. Rebalancing cost example (Section 10.10)

In [18]:
V = 1_000_000
drift_pct = 0.05
trade_size = drift_pct * V
cost_bps = 15
one_way_cost = trade_size * (cost_bps / 10_000)
total_cost = 2 * one_way_cost

print(f"Trade size: ${trade_size:,.0f}")
print(f"One-way cost: ${one_way_cost:,.0f}")
print(f"Total round-trip cost: ${total_cost:,.0f} ({total_cost/V:.4%} of portfolio)")

Trade size: $50,000
One-way cost: $75
Total round-trip cost: $150 (0.0150% of portfolio)


## 11. Robo-advisors: risk-tolerance heuristic vs. mean-variance allocation (Section 10.14)

In [19]:
import numpy as np

age = 35
equity_heuristic = (110 - age) / 100
bond_heuristic = 1 - equity_heuristic
print("110-age heuristic equity weight:", equity_heuristic)
print("110-age heuristic bond weight:", bond_heuristic)

sigma_e, sigma_b, rho = 0.16, 0.05, -0.20
cov_eb = rho * sigma_e * sigma_b
w_e_mv = (sigma_b**2 - cov_eb) / (sigma_e**2 + sigma_b**2 - 2 * cov_eb)
w_b_mv = 1 - w_e_mv
var_mv = w_e_mv**2 * sigma_e**2 + w_b_mv**2 * sigma_b**2 + 2 * w_e_mv * w_b_mv * cov_eb
sigma_mv = np.sqrt(var_mv)

r_e, r_b = 0.08, 0.03
ret_mv = w_e_mv * r_e + w_b_mv * r_b

var_h = equity_heuristic**2 * sigma_e**2 + bond_heuristic**2 * sigma_b**2 + 2 * equity_heuristic * bond_heuristic * cov_eb
sigma_h = np.sqrt(var_h)
ret_h = equity_heuristic * r_e + bond_heuristic * r_b

print("\nminimum-variance equity/bond weights:", round(w_e_mv, 4), round(w_b_mv, 4))
print("minimum-variance volatility:", round(sigma_mv, 4), " expected return:", round(ret_mv, 4))
print("heuristic (75/25) volatility:", round(sigma_h, 4), " expected return:", round(ret_h, 4))
print("heuristic Sharpe (rf=0):", round(ret_h / sigma_h, 4))
print("minimum-variance Sharpe (rf=0):", round(ret_mv / sigma_mv, 4))

110-age heuristic equity weight: 0.75
110-age heuristic bond weight: 0.25

minimum-variance equity/bond weights: 0.131 0.869
minimum-variance volatility: 0.0443  expected return: 0.0365
heuristic (75/25) volatility: 0.1181  expected return: 0.0675
heuristic Sharpe (rf=0): 0.5714
minimum-variance Sharpe (rf=0): 0.825


## 12. Robo-advisors: tax-loss harvesting (Section 10.14)

In [20]:
position_value = 50000
loss_pct = 0.10
loss_amount = position_value * loss_pct

marginal_tax_rate = 0.32
ordinary_income_offset_cap = 3000
ltcg_rate = 0.15

ordinary_offset = min(loss_amount, ordinary_income_offset_cap)
savings_ordinary = ordinary_offset * marginal_tax_rate
remaining_loss = loss_amount - ordinary_offset
savings_gains = remaining_loss * ltcg_rate
total_savings = savings_ordinary + savings_gains

print("loss amount:", loss_amount)
print("savings from offsetting ordinary income:", savings_ordinary)
print("remaining loss offsetting capital gains:", remaining_loss, " savings:", savings_gains)
print("total first-year tax savings:", total_savings)

loss amount: 5000.0
savings from offsetting ordinary income: 960.0
remaining loss offsetting capital gains: 2000.0  savings: 300.0
total first-year tax savings: 1260.0


## 13. Robo-advisors: fee drag over a long horizon (Section 10.14)

In [21]:
initial = 100000
years = 30
gross_return = 0.07
fee_robo = 0.0025
fee_human = 0.01

fv_no_fee = initial * (1 + gross_return) ** years
fv_robo = initial * (1 + gross_return - fee_robo) ** years
fv_human = initial * (1 + gross_return - fee_human) ** years

print("future value, no fee:", round(fv_no_fee, 2))
print("future value, robo fee (25 bps):", round(fv_robo, 2))
print("future value, traditional advisor fee (100 bps):", round(fv_human, 2))
print("robo advantage over traditional advisor:", round(fv_robo - fv_human, 2))

future value, no fee: 761225.5
future value, robo fee (25 bps): 709637.42
future value, traditional advisor fee (100 bps): 574349.12
robo advantage over traditional advisor: 135288.31


## Exercises (match Chapter 10, Suggested Exercises)

Selected exercises reproduced below; use the cells above as templates for the others.

In [22]:
# Exercise 2: two assets, sigX=15%, sigY=25%, rho=0.20
sigX_ex, sigY_ex, rho_ex = 0.15, 0.25, 0.20
sigXY_ex = rho_ex * sigX_ex * sigY_ex
wX_ex = (sigY_ex**2 - sigXY_ex) / (sigX_ex**2 + sigY_ex**2 - 2*sigXY_ex)
wY_ex = 1 - wX_ex
port_std_ex = np.sqrt(wX_ex**2*sigX_ex**2 + wY_ex**2*sigY_ex**2 + 2*wX_ex*wY_ex*sigXY_ex)
print(f"Exercise 2 -- wX={wX_ex:.4%}, wY={wY_ex:.4%}, portfolio std={port_std_ex:.4%}")

# Exercise 6: risk parity for 3 assets with vols 10%, 20%, 30%
vols_ex6 = np.array([0.10, 0.20, 0.30])
w_rp_ex6 = (1/vols_ex6) / np.sum(1/vols_ex6)
print(f"Exercise 6 -- risk parity weights: {w_rp_ex6}")

# Exercise 11: tracking error for a different portfolio/benchmark pair
port_ex11 = np.array([0.015, -0.005, 0.020, 0.005])
bench_ex11 = np.array([0.010, 0.005, 0.010, 0.005])
active_ex11 = port_ex11 - bench_ex11
te_ex11 = active_ex11.std(ddof=1)
ir_ex11 = active_ex11.mean() / te_ex11
print(f"Exercise 11 -- tracking error={te_ex11:.4%}, information ratio={ir_ex11:.4f}")

# Exercise 12: currency example, yen appreciates
local_ex12 = 0.02
fx_ex12 = 0.04
unhedged_ex12 = (1+local_ex12)*(1+fx_ex12) - 1
print(f"Exercise 12 -- unhedged USD return: {unhedged_ex12:.4%}")

Exercise 2 -- wX=78.5714%, wY=21.4286%, portfolio std=13.8873%
Exercise 6 -- risk parity weights: [0.54545455 0.27272727 0.18181818]
Exercise 11 -- tracking error=0.8539%, information ratio=0.1464
Exercise 12 -- unhedged USD return: 6.0800%
